# Part 5 - LightGBM Competition Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

TARGET='addicted_label'
ID='id'

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES].copy()
y=train_fe[TARGET]
X_test=test_fe[FEATURES].copy()

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

for c in cat_cols:
    X[c]=X[c].astype('category')
    X_test[c]=X_test[c].astype('category')


## Parameters

In [ ]:
params=dict(
    objective='binary',
    metric='auc',
    n_estimators=5000,
    learning_rate=0.02,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1
)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
pred=np.zeros(len(X_test))
scores=[]
fi=pd.DataFrame(index=FEATURES)


## Cross Validation

In [ ]:
for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print("="*60)
    print("Fold",fold)

    model=LGBMClassifier(**params)

    model.fit(
        X.iloc[tr],y.iloc[tr],
        eval_set=[(X.iloc[va],y.iloc[va])],
        eval_metric='auc',
        categorical_feature=cat_cols,
        callbacks=[]
    )

    val=model.predict_proba(X.iloc[va])[:,1]
    oof[va]=val

    auc=roc_auc_score(y.iloc[va],val)
    scores.append(auc)

    pred+=model.predict_proba(X_test)[:,1]/cv.n_splits

    fi[f'fold_{fold}']=model.feature_importances_

    print(f"Fold {fold} AUC: {auc:.6f}")

print("\nMean CV:",np.mean(scores))
print("Overall:",roc_auc_score(y,oof))


## Feature Importance

In [ ]:
fi['mean']=fi.mean(axis=1)
fi=fi.sort_values('mean',ascending=False)

display(fi.head(20))

plt.figure(figsize=(8,8))
fi['mean'].head(20).sort_values().plot.barh()
plt.title("Top 20 LightGBM Features")
plt.tight_layout()
plt.show()


## Submission

In [ ]:
submission=sample.copy()
submission[TARGET]=pred
submission.to_csv("submission_lightgbm.csv",index=False)

display(submission.head())
print("Saved submission_lightgbm.csv")


### Next

Part 6 will include:
- XGBoost with tuned parameters
- OOF prediction saving
- Preparation for weighted blending with CatBoost and LightGBM.
